# Modelo Predictivo de Demanda — NYC TLC

**Objetivo:** predecir la demanda diaria de viajes (yellow ta como guíaxi) a 90 días usando Facebook Prophet, con evaluación cuantitativa del error antes de generar la predicción final.

## Setup

Configuración inicial: sesión de Spark y silenciado de logs verbosos de `cmdstanpy` (motor de optimización interno de Prophet). Estos logs son informativos, no errores, pero por defecto son muy detallados.


In [1]:
!pip install plotly --quiet

In [3]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

from spark_utils import crear_spark_session

spark = crear_spark_session()

RUTA_SERIE_TIEMPO = "/home/jovyan/data/gold/predictivo/features_serie_tiempo/tipo_vehiculo=yellow/*/*.parquet"
RUTA_DESTINO_GOLD = "/home/jovyan/data/gold/predictivo/forecast_demanda"
HORIZONTE_DIAS = 90
TIPO_VEHICULO = "yellow"

print("Completo")

2026-07-10 16:53:53,027 [INFO] SparkSession creada: 3.5.0


Completo


## Fase 0 — Auditoría de calidad de datos

Antes de entrenar cualquier modelo, se inspecciona la serie completa en busca de valores extremos que puedan distorsionar el ajuste. Se calculan estadísticos descriptivos y se listan los días de menor volumen.


In [4]:
df_spark = spark.read.parquet(RUTA_SERIE_TIEMPO).orderBy("fecha")

df_spark.select(
    F.min("total_viajes").alias("min_viajes"),
    F.max("total_viajes").alias("max_viajes"),
    F.avg("total_viajes").alias("promedio"),
    F.stddev("total_viajes").alias("desviacion")
).show()

# Los 10 días con menor volumen — para identificar outliers
df_spark.orderBy("total_viajes").select("fecha", "total_viajes", "dia_semana").show(10)

+----------+----------+------------------+-----------------+
|min_viajes|max_viajes|          promedio|       desviacion|
+----------+----------+------------------+-----------------+
|      2567|    173990|110157.05291970803|19905.01348665825|
+----------+----------+------------------+-----------------+

+----------+------------+----------+
|     fecha|total_viajes|dia_semana|
+----------+------------+----------+
|2023-09-22|        2567|         6|
|2023-09-24|        2985|         1|
|2023-09-23|        3473|         7|
|2023-09-21|       38644|         5|
|2023-12-25|       43345|         2|
|2024-12-25|       52070|         4|
|2023-07-04|       55860|         3|
|2024-07-04|       59202|         5|
|2023-11-23|       59828|         5|
|2023-07-03|       62427|         2|
+----------+------------+----------+
only showing top 10 rows



In [5]:
FECHAS_EXCLUIR = ['2023-09-22', '2023-09-23', '2023-09-24']
print(f"Fechas excluidas por falla de ingesta confirmada: {FECHAS_EXCLUIR}")

Fechas excluidas por falla de ingesta confirmada: ['2023-09-22', '2023-09-23', '2023-09-24']


## Fase 1 — Train / Test Split temporal

Al tratarse de una serie de tiempo, el split debe ser **cronológico**, no aleatorio: un split aleatorio filtraría información del futuro hacia el entrenamiento (data leakage). Se reservan los últimos 90 días como conjunto de pruebro.


In [6]:
corte = df_spark.select(
    F.date_sub(F.max("fecha"), HORIZONTE_DIAS).alias("corte")
).collect()[0][0]

train_spark = df_spark.filter(F.col("fecha") <= corte)
test_spark = df_spark.filter(F.col("fecha") > corte)

train_pd = (
    train_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
    .orderBy("ds")
    .toPandas()
)
train_pd['ds'] = train_pd['ds'].astype('datetime64[ns]')
train_pd = train_pd[~train_pd['ds'].astype(str).isin(FECHAS_EXCLUIR)].reset_index(drop=True)

test_pd = (
    test_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
    .orderBy("ds")
    .toPandas()
)
test_pd['ds'] = test_pd['ds'].astype('datetime64[ns]')

print(f"Train: {len(train_pd)} días ({train_pd['ds'].min().date()} a {train_pd['ds'].max().date()})")
print(f"Test:  {len(test_pd)} días ({test_pd['ds'].min().date()} a {test_pd['ds'].max().date()})")

Train: 1003 días (2023-01-01 a 2025-10-02)
Test:  90 días (2025-10-03 a 2025-12-31)


## Fase 2 — Modelo baseline

Se entrena un primer modelo Prophet con estacionalidad anual y semanal (sin estacionalidad diaria, ya que los datos están agregados a nivel de día.

In [7]:
modelo_baseline = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
modelo_baseline.fit(train_pd)

futuro_baseline = modelo_baseline.make_future_dataframe(periods=HORIZONTE_DIAS)
forecast_baseline = modelo_baseline.predict(futuro_baseline)

print("Modelo baseline entrenado")
forecast_baseline[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(5)

Modelo baseline entrenado


,ds,yhat,yhat_lower,yhat_upper
1088,2025-12-27,115989.002411,96369.090273,136099.427691
1089,2025-12-28,96876.591115,76850.280953,116857.497070
1090,2025-12-29,90650.702808,69994.140578,109476.712977
1091,2025-12-30,104149.552370,85167.962575,124226.925824
1092,2025-12-31,109816.281596,90062.338695,130551.434522


## Fase 3 — Evaluación cuantitativa (baseline)

Se compara la predicción contra los valores reales del conjunto de prueba usando tres métricas estándar:

- **MAE** (Error Absoluto Medio): magnitud promedio del error, en viajes.
- **MAPE** (Error Porcentual Absoluto Medio): el más interpretable — "el modelo se equivoca en promedio X%". <10% se considera bueno, 10–20% aceptable, >20% amerita revisión.
- **RMSE** (Raíz del Error Cuadrático Medio): penaliza más los errores grandes; útil para detectar si hay días puntuales con fallos importantes.

In [8]:
def evaluar(forecast, test, nombre):
    comparacion = forecast[['ds', 'yhat']].merge(test[['ds', 'y']], on='ds')
    mae = mean_absolute_error(comparacion['y'], comparacion['yhat'])
    mape = mean_absolute_percentage_error(comparacion['y'], comparacion['yhat']) * 100
    rmse = np.sqrt(mean_squared_error(comparacion['y'], comparacion['yhat']))
    print(f"[{nombre}] MAE: {mae:,.0f} | MAPE: {mape:.2f}% | RMSE: {rmse:,.0f}")
    return {"modelo": nombre, "mae": mae, "mape": mape, "rmse": rmse}

metricas_baseline = evaluar(forecast_baseline, test_pd, "Baseline")

[Baseline] MAE: 12,249 | MAPE: 11.23% | RMSE: 17,455


## Fase 4 — Mejora: feriados como regresor

La demanda de taxis en NYC cambia notablemente en feriados (Navidad, Año Nuevo, Thanksgiving, 4 de julio). Prophet permite incorporar el calendario de feriados de un país directamente como regresor, sin necesidad de features manuales.

Se entrena un segundo modelo con esta mejora y se compara contra el baseline usando las mismas métricas.

In [10]:
modelo_holidays = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
modelo_holidays.add_country_holidays(country_name='US')
modelo_holidays.fit(train_pd)

futuro_holidays = modelo_holidays.make_future_dataframe(periods=HORIZONTE_DIAS)
forecast_holidays = modelo_holidays.predict(futuro_holidays)

metricas_holidays = evaluar(forecast_holidays, test_pd, "Con feriados")

[Con feriados] MAE: 11,203 | MAPE: 9.88% | RMSE: 15,211


### Comparación
| Métrica | Baseline | Con feriados | Mejora |
|---|---|---|---|
| MAE | 12,249 | 11,203 | -8.5% |
| MAPE | 11.23% | 9.88% | -1.35 pp |
| RMSE | 17,455 | 15,211 | -12.

Las tres métricas mejoran de forma consistente. El RMSE mejora proporcionalmente más que el MAE, lo que indica que buena parte del error del modelo baseline se concentraba en días puntuales (feriados) que el modelo no reconocía — exactamente lo que se buscaba corregir.8% |

## Fase 5 — Reentrenamiento final con el histórico completo

Una vez validado el modelo (Fases 3–4) sobre el conjunto de prueba, se reentrena con el **100% de los datos históricos disponibles** (incluyendo el tramo que antes se usó como test) para generar la predicción real hacia el futuro. Esto maximiza la información disponible para el modelo de producción.

In [13]:
df_completo_pd = (
    df_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
    .orderBy("ds")
    .toPandas()
)
df_completo_pd['ds'] = df_completo_pd['ds'].astype('datetime64[ns]')
df_completo_pd = df_completo_pd[~df_completo_pd['ds'].astype(str).isin(FECHAS_EXCLUIR)].reset_index(drop=True)

modelo_produccion = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
modelo_produccion.add_country_holidays(country_name='US')
modelo_produccion.fit(df_completo_pd)

futuro_real = modelo_produccion.make_future_dataframe(periods=HORIZONTE_DIAS)
forecast_real = modelo_produccion.predict(futuro_real)

print(f"Predicción generada hasta: {forecast_real['ds'].max().date()}")
forecast_real[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(5)

Predicción generada hasta: 2026-03-31


,ds,yhat,yhat_lower,yhat_upper
1178,2026-03-27,137836.970060,120149.809159,156314.928278
1179,2026-03-28,140492.948049,121052.636966,157960.198687
1180,2026-03-29,123618.282577,105083.250525,142169.899674
1181,2026-03-30,120645.577520,100954.209976,139755.657872
1182,2026-03-31,133877.667428,115676.518653,152521.840194


## Fase 6 — Exportación a Gold

Se exporta el forecast como tabla Parquet particionada por `tipo_vehiculo`, siguiendo el mismo patrón de las demás tablas Gold del proyecto (`descriptivo.py`, `diagnostico.py`).

La columna `es_prediccion` (booleana) permite en Power BI diferenciar visualmente la serie histórica de la proyección futura dentro del mismo visual.


In [15]:
forecast_export = forecast_real[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_export = forecast_export.rename(columns={
    'ds': 'fecha',
    'yhat': 'prediccion',
    'yhat_lower': 'prediccion_min',
    'yhat_upper': 'prediccion_max'
})

fecha_max_historica = df_completo_pd['ds'].max()
forecast_export['es_prediccion'] = forecast_export['fecha'] > fecha_max_historica
forecast_export['tipo_vehiculo'] = TIPO_VEHICULO

print(f"Fecha máxima histórica: {fecha_max_historica.date()}")
print(f"Filas históricas: {(~forecast_export['es_prediccion']).sum()}")
print(f"Filas de predicción: {forecast_export['es_prediccion'].sum()}")

forecast_spark = spark.createDataFrame(forecast_export)

forecast_spark.write.mode("overwrite").partitionBy("tipo_vehiculo").parquet(RUTA_DESTINO_GOLD)

print(f"Forecast exportado: {forecast_spark.count()} filas -> {RUTA_DESTINO_GOLD}")

Fecha máxima histórica: 2025-12-31
Filas históricas: 1093
Filas de predicción: 90
Forecast exportado: 1183 filas -> /home/jovyan/data/gold/predictivo/forecast_demanda


## Proceso de los demás tipos de Vehículos

In [19]:
def entrenar_y_exportar(spark, tipo, fechas_excluir=None, usar_feriados=True):
    fechas_excluir = fechas_excluir or []
    print(f"\n=== {tipo.upper()} ===")
    
    df_spark = spark.read.parquet(RUTA_BASE.format(tipo=tipo)).orderBy("fecha")
    
    df_completo_pd = (
        df_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
        .orderBy("ds").toPandas()
    )
    df_completo_pd['ds'] = df_completo_pd['ds'].astype('datetime64[ns]')
    
    if fechas_excluir:
        df_completo_pd = df_completo_pd[~df_completo_pd['ds'].astype(str).isin(fechas_excluir)].reset_index(drop=True)
    
    modelo_final = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                           daily_seasonality=False, changepoint_prior_scale=0.05, interval_width=0.95)
    
    if usar_feriados:
        modelo_final.add_country_holidays(country_name='US')
        
    modelo_final.fit(df_completo_pd)
    forecast_final = modelo_final.predict(modelo_final.make_future_dataframe(periods=HORIZONTE_DIAS))
    
    forecast_export = forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_export = forecast_export.rename(columns={
        'ds': 'fecha',
        'yhat': 'prediccion',
        'yhat_lower': 'prediccion_min', 
        'yhat_upper': 'prediccion_max'
    })
    
    fecha_max = df_completo_pd['ds'].max()
    forecast_export['es_prediccion'] = forecast_export['fecha'] > fecha_max
    forecast_export['tipo_vehiculo'] = tipo
    
    forecast_spark = spark.createDataFrame(forecast_export)
    forecast_spark.write.mode("append").partitionBy("tipo_vehiculo").parquet(RUTA_DESTINO_GOLD)
    print(f"  Exportado: {forecast_spark.count()} filas -> tipo_vehiculo={tipo}")

# Ejecutar para los 3 tipos
for tipo in ["green", "fhv", "fhvhv"]:
    entrenar_y_exportar(spark, tipo, fechas_excluir=None, usar_feriados=True)


=== GREEN ===
  Exportado: 1186 filas -> tipo_vehiculo=green

=== FHV ===
  Exportado: 1186 filas -> tipo_vehiculo=fhv

=== FHVHV ===
  Exportado: 1186 filas -> tipo_vehiculo=fhvhv
